In [ ]:
import pygame
import time
import tictactoe as ttt

pygame.init()

WIDTH, HEIGHT = 700, 500
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Minimax TicTacToe - Unbeatable AI")

BLACK = (10, 10, 20)
WHITE = (245, 245, 255)
GRAY = (100, 100, 120)
LIGHT_GRAY = (180, 180, 200)
BLUE = (60, 120, 220)
GREEN = (60, 200, 100)
DARK_BLUE = (30, 70, 150)
PURPLE = (150, 60, 200)
ORANGE = (255, 140, 50)
BG_COLOR = (20, 25, 45)

try:
    font_small = pygame.font.Font("OpenSans-Regular.ttf", 24)
    font_medium = pygame.font.Font("OpenSans-Regular.ttf", 32)
    font_large = pygame.font.Font("OpenSans-Regular.ttf", 48)
except:
    font_small = pygame.font.SysFont("arial", 24)
    font_medium = pygame.font.SysFont("arial", 32)
    font_large = pygame.font.SysFont("arial", 48)

def draw_button(screen, text, rect, color, hover_color, is_hovered):
    color_to_use = hover_color if is_hovered else color
    pygame.draw.rect(screen, color_to_use, rect, border_radius=12)
    pygame.draw.rect(screen, WHITE, rect, 2, border_radius=12)
    text_surface = font_medium.render(text, True, BLACK)
    text_rect = text_surface.get_rect(center=rect.center)
    screen.blit(text_surface, text_rect)

def draw_board(board, tiles, screen):
    for i in range(3):
        for j in range(3):
            rect = tiles[i][j]
            if board[i][j] == ttt.X:
                margin = 15
                start_pos1 = (rect.left + margin, rect.top + margin)
                end_pos1 = (rect.right - margin, rect.bottom - margin)
                start_pos2 = (rect.right - margin, rect.top + margin)
                end_pos2 = (rect.left + margin, rect.bottom - margin)
                pygame.draw.line(screen, BLUE, start_pos1, end_pos1, 6)
                pygame.draw.line(screen, BLUE, start_pos2, end_pos2, 6)
            elif board[i][j] == ttt.O:
                center = rect.center
                radius = rect.width // 2 - 15
                pygame.draw.circle(screen, ORANGE, center, radius, 6)

def draw_glow_effect(screen, rect, color):
    for i in range(3, 0, -1):
        alpha_rect = rect.inflate(i * 4, i * 4)
        glow_surface = pygame.Surface(alpha_rect.size, pygame.SRCALPHA)
        pygame.draw.rect(glow_surface, (*color, 50 // i), glow_surface.get_rect(), border_radius=12)
        screen.blit(glow_surface, alpha_rect.topleft)

def main():
    clock = pygame.time.Clock()
    user = None
    board = ttt.initial_state()
    ai_turn = False
    game_over_animation = 0
    winner_effect = None
    running = True

    while running:

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        screen.fill(BG_COLOR)

        for i in range(1, 3):
            pygame.draw.line(screen, GRAY, (WIDTH//3 * i, 50), (WIDTH//3 * i, HEIGHT-50), 1)
            pygame.draw.line(screen, GRAY, (0, HEIGHT//3 * i + 30), (WIDTH, HEIGHT//3 * i + 30), 1)

        if user is None:
            title = font_large.render("MINIMAX TIC TAC TOE", True, WHITE)
            title_rect = title.get_rect(center=(WIDTH / 2, 70))
            screen.blit(title, title_rect)
            
            subtitle = font_small.render("You vs Unbeatable AI", True, LIGHT_GRAY)
            subtitle_rect = subtitle.get_rect(center=(WIDTH / 2, 115))
            screen.blit(subtitle, subtitle_rect)

            playXButton = pygame.Rect(WIDTH // 4 - 100, HEIGHT // 2 - 30, 180, 70)
            playOButton = pygame.Rect(3 * WIDTH // 4 - 80, HEIGHT // 2 - 30, 180, 70)

            mouse_x, mouse_y = pygame.mouse.get_pos()
            hover_x = playXButton.collidepoint(mouse_x, mouse_y)
            hover_o = playOButton.collidepoint(mouse_x, mouse_y)

            draw_button(screen, "Play as X", playXButton, BLUE, DARK_BLUE, hover_x)
            draw_button(screen, "Play as O", playOButton, ORANGE, (200, 100, 30), hover_o)

            info = font_small.render("You go first as X | AI goes first as O", True, LIGHT_GRAY)
            info_rect = info.get_rect(center=(WIDTH / 2, HEIGHT - 50))
            screen.blit(info, info_rect)

            click, _, _ = pygame.mouse.get_pressed()
            if click == 1:
                if hover_x:
                    time.sleep(0.15)
                    user = ttt.X
                elif hover_o:
                    time.sleep(0.15)
                    user = ttt.O
                    ai_turn = True

        else:
            tile_size = 100
            board_width = tile_size * 3
            start_x = (WIDTH - board_width) // 2
            start_y = (HEIGHT - board_width) // 2 + 20

            tiles = []
            for i in range(3):
                row = []
                for j in range(3):
                    rect = pygame.Rect(
                        start_x + j * tile_size,
                        start_y + i * tile_size,
                        tile_size, tile_size
                    )
                    pygame.draw.rect(screen, (30, 35, 55), rect, border_radius=10)
                    pygame.draw.rect(screen, WHITE, rect, 2, border_radius=10)
                    
                    if board[i][j] == ttt.EMPTY and user == ttt.player(board) and not ttt.terminal(board):
                        mouse_x, mouse_y = pygame.mouse.get_pos()
                        if rect.collidepoint(mouse_x, mouse_y):
                            pygame.draw.rect(screen, (60, 65, 85), rect, border_radius=10)
                    
                    row.append(rect)
                tiles.append(row)

            draw_board(board, tiles, screen)

            game_over = ttt.terminal(board)
            current_player = ttt.player(board)

            if game_over:
                winner_player = ttt.winner(board)
                if winner_player is None:
                    status_text = "GAME OVER - TIE!"
                    status_color = LIGHT_GRAY
                else:
                    status_text = f"{winner_player} WINS!"
                    status_color = BLUE if winner_player == ttt.X else ORANGE
                    if winner_effect is None:
                        winner_effect = winner_player
                        game_over_animation = 20
                
                if game_over_animation > 0:
                    game_over_animation -= 1
                    
            elif user == current_player:
                status_text = f"Your turn ({user})"
                status_color = BLUE if user == ttt.X else ORANGE
            else:
                status_text = "AI is thinking..."
                status_color = PURPLE

            if game_over_animation > 0:
                scale = 1 + (game_over_animation / 20) * 0.3
                animated_font = pygame.font.Font(None, int(48 * scale))
                status_surface = animated_font.render(status_text, True, status_color)
            else:
                status_surface = font_large.render(status_text, True, status_color)
            
            status_rect = status_surface.get_rect(center=(WIDTH / 2, 50))
            screen.blit(status_surface, status_rect)

            score_font = pygame.font.Font(None, 24)
            score_text = score_font.render("Can you beat the AI? (Spoiler: You can't)", True, LIGHT_GRAY)
            score_rect = score_text.get_rect(center=(WIDTH / 2, HEIGHT - 25))
            screen.blit(score_text, score_rect)

            if user != current_player and not game_over:
                if ai_turn:
                    time.sleep(0.4)
                    move = ttt.minimax(board)
                    if move is not None:
                        board = ttt.result(board, move)
                    ai_turn = False
                else:
                    ai_turn = True

            click, _, _ = pygame.mouse.get_pressed()
            if click == 1 and user == current_player and not game_over:
                mouse_x, mouse_y = pygame.mouse.get_pos()
                for i in range(3):
                    for j in range(3):
                        if (board[i][j] == ttt.EMPTY and tiles[i][j].collidepoint(mouse_x, mouse_y)):
                            board = ttt.result(board, (i, j))
                            ai_turn = True
                            time.sleep(0.1)

            if game_over:
                againButton = pygame.Rect(WIDTH // 2 - 100, HEIGHT - 85, 200, 50)
                mouse_x, mouse_y = pygame.mouse.get_pos()
                hover_again = againButton.collidepoint(mouse_x, mouse_y)
                
                draw_glow_effect(screen, againButton, GREEN)
                draw_button(screen, "PLAY AGAIN", againButton, GREEN, (40, 150, 60), hover_again)
                
                click, _, _ = pygame.mouse.get_pressed()
                if click == 1 and hover_again:
                    time.sleep(0.2)
                    user = None
                    board = ttt.initial_state()
                    ai_turn = False
                    game_over_animation = 0
                    winner_effect = None

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()

if __name__ == "__main__":
    main()